In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)


TensorFlow: 2.19.0


In [ ]:
DATA_ROOT = Path("/content/drive/MyDrive/HAM10000")

IMG_DIRS = [
    DATA_ROOT / "HAM10000_images_part_1",
    DATA_ROOT / "HAM10000_images_part_2",
]

CSV_PATH = DATA_ROOT / "HAM10000_metadata.csv"

IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 25
SEED = 42


In [ ]:
df = pd.read_csv(CSV_PATH)

# Keep only needed columns
df = df[["image_id", "dx"]]

print("Class distribution:")
print(df["dx"].value_counts())


Class distribution:
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64


In [ ]:
class_names = sorted(df["dx"].unique())
class_to_index = {name: i for i, name in enumerate(class_names)}

df["label"] = df["dx"].map(class_to_index)

num_classes = len(class_names)

print("Classes:", class_names)


Classes: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']


In [ ]:
def find_image_path(image_id):
    for folder in IMG_DIRS:
        path = folder / f"{image_id}.jpg"
        if path.exists():
            return str(path)
    return None

df["image_path"] = df["image_id"].apply(find_image_path)
df = df.dropna()

print("Total usable images:", len(df))


Total usable images: 10015


In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=SEED
)


In [ ]:
def safe_load_image(path, label):
    def _load(path_str):
        try:
            img = tf.io.read_file(path_str)
            img = tf.image.decode_jpeg(img, channels=3)
            img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
            img = tf.cast(img, tf.float32)
            return img
        except:
            # Return black image if corrupted
            return tf.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32)

    img = tf.py_function(
        func=_load,
        inp=[path],
        Tout=tf.float32
    )

    img.set_shape((IMG_SIZE, IMG_SIZE, 3))
    return img, label


In [ ]:
base = keras.applications.ConvNeXtTiny(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base.trainable = True  # FULL FINE-TUNING

inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
x = keras.applications.convnext.preprocess_input(inputs)
x = base(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)

outputs = layers.Dense(num_classes, activation="softmax")(x)
model = keras.Model(inputs, outputs)

model.summary()


Model: "functional_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_17 (InputLayer)     │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ convnext_tiny (Functional)      │ (None, 7, 7, 768)      │    27,820,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 768)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 768)            │         3,072 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 7)              │         5,383 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,828,583 (106.16 MB)

 Trainable params: 27,827,047 (106.15 MB)

 Non-trainable params: 1,536 (6.00 KB)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(3e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        "accuracy",
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3_accuracy"
        )
    ]
)


In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)


Epoch 1/25
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step - accuracy: 0.5671 - loss: 1.5636 - top3_accuracy: 0.7805

InvalidArgumentError: Graph execution error:

Detected at node DecodeJpeg defined at (most recent call last):
<stack traces unavailable>
Detected at node DecodeJpeg defined at (most recent call last):
<stack traces unavailable>
2 root error(s) found.
  (0) INVALID_ARGUMENT:  Input is empty.
	 [[{{node DecodeJpeg}}]]
	 [[IteratorGetNext]]
	 [[IteratorGetNext/_4]]
  (1) INVALID_ARGUMENT:  Input is empty.
	 [[{{node DecodeJpeg}}]]
	 [[IteratorGetNext]]
0 successful operations.
0 derived errors ignored. [Op:__inference_multi_step_on_iterator_120854]

In [ ]:
# ===============================
# FULL HAM10000 TRAINING SCRIPT
# ===============================



# ---- IMPORTS ----
import tensorflow as tf
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers
from google.colab import drive

print("TensorFlow version:", tf.__version__)

# ---- MOUNT DRIVE ----
drive.mount("/content/drive")

# ---- CONFIG ----
DATA_ROOT = Path("/content/drive/MyDrive/HAM10000")
IMG_DIRS = [
    DATA_ROOT / "HAM10000_images_part_1",
    DATA_ROOT / "HAM10000_images_part_2",
]
CSV_PATH = DATA_ROOT / "HAM10000_metadata.csv"

IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 25
SEED = 42

# ---- LOAD METADATA ----
df = pd.read_csv(CSV_PATH)[["image_id", "dx"]]
print("\nClass distribution:\n", df["dx"].value_counts())

# ---- ENCODE LABELS ----
class_names = sorted(df["dx"].unique())
class_to_index = {c: i for i, c in enumerate(class_names)}
df["label"] = df["dx"].map(class_to_index)
num_classes = len(class_names)

print("\nClasses:", class_names)

# ---- RESOLVE IMAGE PATHS ----
def find_image_path(image_id):
    for d in IMG_DIRS:
        p = d / f"{image_id}.jpg"
        if p.exists():
            return str(p)
    return None

df["image_path"] = df["image_id"].apply(find_image_path)
df = df.dropna()
print("Total usable images:", len(df))

# ---- TRAIN / VAL SPLIT ----
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=SEED
)

# ---- SAFE IMAGE LOADER ----
def safe_load_image(path, label):
    def _load(path_str):
        try:
            img = tf.io.read_file(path_str)
            img = tf.image.decode_jpeg(img, channels=3)
            img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
            img = tf.cast(img, tf.float32)
            return img
        except:
            return tf.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32)

    img = tf.py_function(_load, [path], tf.float32)
    img.set_shape((IMG_SIZE, IMG_SIZE, 3))
    return img, label

# ---- DATASET PIPELINE ----
def make_dataset(dataframe, training=True):
    ds = tf.data.Dataset.from_tensor_slices(
        (dataframe["image_path"].values, dataframe["label"].values)
    )
    ds = ds.map(safe_load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.apply(tf.data.experimental.ignore_errors())
    if training:
        ds = ds.shuffle(1024)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)

# ---- MODEL ----
base = keras.applications.ConvNeXtTiny(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base.trainable = True

inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
x = keras.applications.convnext.preprocess_input(inputs)
x = base(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.summary()

# ---- COMPILE ----
model.compile(
    optimizer=keras.optimizers.Adam(3e-4),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        "accuracy",
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3, name="top3_accuracy"
        )
    ]
)

# ---- TRAIN ----
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

# ---- FINAL EVALUATION ----
print("\nFinal validation metrics:")
results = model.evaluate(val_ds, verbose=0)
for n, v in zip(model.metrics_names, results):
    print(f"{n}: {v:.4f}")

# ---- SAVE MODEL ----
model.save("/content/drive/MyDrive/ham10000_convnext_tiny.keras")
print("\n✅ MODEL SAVED SUCCESSFULLY")


TensorFlow version: 2.19.0
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Class distribution:
 dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64

Classes: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']


Instructions for updating:
Use `tf.data.Dataset.ignore_errors` instead.


Total usable images: 10015


Model: "functional_19"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_23 (InputLayer)     │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ convnext_tiny (Functional)      │ (None, 7, 7, 768)      │    27,820,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 768)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 768)            │         3,072 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │         5,383 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

 Trainable params: 27,827,047 (106.15 MB)

 Non-trainable params: 1,536 (6.00 KB)

Epoch 1/25
    501/Unknown 111s 124ms/step - accuracy: 0.5693 - loss: 1.4953 - top3_accuracy: 0.7802

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


501/501 ━━━━━━━━━━━━━━━━━━━━ 214s 329ms/step - accuracy: 0.5695 - loss: 1.4945 - top3_accuracy: 0.7804 - val_accuracy: 0.7124 - val_loss: 1.3628 - val_top3_accuracy: 0.8248
Epoch 2/25
501/501 ━━━━━━━━━━━━━━━━━━━━ 47s 90ms/step - accuracy: 0.7827 - loss: 0.6225 - top3_accuracy: 0.9624 - val_accuracy: 0.7429 - val_loss: 1.0106 - val_top3_accuracy: 0.9586
Epoch 3/25
501/501 ━━━━━━━━━━━━━━━━━━━━ 47s 90ms/step - accuracy: 0.8400 - loss: 0.4485 - top3_accuracy: 0.9827 - val_accuracy: 0.5662 - val_loss: 1.2784 - val_top3_accuracy: 0.9311
Epoch 4/25
501/501 ━━━━━━━━━━━━━━━━━━━━ 47s 90ms/step - accuracy: 0.9024 - loss: 0.2739 - top3_accuracy: 0.9934 - val_accuracy: 0.7853 - val_loss: 0.9475 - val_top3_accuracy: 0.9261
Epoch 5/25
501/501 ━━━━━━━━━━━━━━━━━━━━ 48s 90ms/step - accuracy: 0.9528 - loss: 0.1493 - top3_accuracy: 0.9984 - val_accuracy: 0.7728 - val_loss: 1.1506 - val_top3_accuracy: 0.9131
Epoch 6/25
501/501 ━━━━━━━━━━━━━━━━━━━━ 47s 90ms/step - accuracy: 0.9657 - loss: 0.0924 - top3_accu

In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
from pathlib import Path
from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow:", tf.__version__)

# ============================================================
# 2. CONFIG
# ============================================================
DATASET_ROOT = Path("/content/drive/MyDrive/archive (2)/Dataset")
TRAIN_DIR = DATASET_ROOT / "train"

IMG_SIZE = 224
BATCH_SIZE = 8          # 🔥 Safe for EfficientNetV2-L
EPOCHS_PHASE1 = 15      # with class weights
EPOCHS_PHASE2 = 10      # without class weights
SEED = 42

# ============================================================
# 3. LOAD DATASET (INTEGER LABELS FIRST)
# ============================================================
train_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int",
    validation_split=0.2,
    subset="training",
    seed=SEED
)

val_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int",
    validation_split=0.2,
    subset="validation",
    seed=SEED
)

CLASS_NAMES = train_ds.class_names
NUM_CLASSES = len(CLASS_NAMES)
print("Classes:", CLASS_NAMES)

# ============================================================
# 4. ONE-HOT LABELS (REQUIRED FOR LABEL SMOOTHING + TOP-3)
# ============================================================
def one_hot(images, labels):
    return images, tf.one_hot(labels, NUM_CLASSES)

train_ds = train_ds.map(one_hot, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.map(one_hot, num_parallel_calls=tf.data.AUTOTUNE)

# ============================================================
# 5. CLASS WEIGHTS (CAPPED)
# ============================================================
raw_labels = np.concatenate([
    np.argmax(y.numpy(), axis=1)
    for _, y in train_ds
])

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(raw_labels),
    y=raw_labels
)

MAX_WEIGHT = 5.0
CLASS_WEIGHTS = {i: float(min(w, MAX_WEIGHT)) for i, w in enumerate(weights)}
print("Class weights:", CLASS_WEIGHTS)

# ============================================================
# 6. DATA AUGMENTATION
# ============================================================
augmenter = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.05),
])

def augment(images, labels):
    return augmenter(images, training=True), labels

train_ds = train_ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

# ============================================================
# 7. MODEL — EfficientNetV2-L (STRONGER THAN CONVNEXT)
# ============================================================
base = keras.applications.EfficientNetV2L(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base.trainable = True  # FULL FINE-TUNING

inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
x = keras.applications.efficientnet_v2.preprocess_input(inputs)
x = base(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.summary()

# ============================================================
# 8. CALLBACKS
# ============================================================
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_top3_accuracy",
        patience=6,
        restore_best_weights=True,
        mode="max"
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6
    )
]

# ============================================================
# 9. PHASE 1 — WITH CLASS WEIGHTS
# ============================================================
model.compile(
    optimizer=keras.optimizers.Adam(3e-4),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=[
        "accuracy",
        keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_accuracy")
    ]
)

print("\n===== PHASE 1: WITH CLASS WEIGHTS =====")
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    class_weight=CLASS_WEIGHTS,
    callbacks=callbacks
)

# ============================================================
# 10. PHASE 2 — NO CLASS WEIGHTS (CONFIDENCE BOOST)
# ============================================================
model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=[
        "accuracy",
        keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_accuracy")
    ]
)

print("\n===== PHASE 2: NO CLASS WEIGHTS =====")
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE2,
    callbacks=callbacks
)

# ============================================================
# 11. SAVE MODEL
# ============================================================
model.save("/content/drive/MyDrive/efficientnetv2L_skin_top3.keras")

print("\n🎉 DONE — EfficientNetV2-L trained & saved successfully!")


TensorFlow: 2.19.0
Found 2609 files belonging to 19 classes.
Using 2088 files for training.
Found 2609 files belonging to 19 classes.
Using 521 files for validation.
Classes: ['Acne and Rosacea Photos', 'Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions', 'Atopic Dermatitis Photos', 'Cellulitis Impetigo and other Bacterial Infections', 'Eczema Photos', 'Exanthems and Drug Eruptions', 'Herpes HPV and other STDs Photos', 'Light Diseases and Disorders of Pigmentation', 'Lupus and other Connective Tissue diseases', 'Melanoma Skin Cancer Nevi and Moles', 'Poison Ivy Photos and other Contact Dermatitis', 'Psoriasis pictures Lichen Planus and related diseases', 'Seborrheic Keratoses and other Benign Tumors', 'Systemic Disease', 'Tinea Ringworm Candidiasis and other Fungal Infections', 'Urticaria Hives', 'Vascular Tumors', 'Vasculitis Photos', 'Warts Molluscum and other Viral Infections']
Class weights: {0: 0.16043027276219746, 1: 0.4413443246670894, 2: 1.3736842105263158, 3: 

Model: "functional_21"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_26 (InputLayer)     │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-l (Functional)   │ (None, 7, 7, 1280)     │   117,746,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_4      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 19)             │        24,339 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 117,776,307 (449.28 MB)

 Trainable params: 117,261,171 (447.32 MB)

 Non-trainable params: 515,136 (1.97 MB)


===== PHASE 1: WITH CLASS WEIGHTS =====
Epoch 1/15
261/261 ━━━━━━━━━━━━━━━━━━━━ 427s 361ms/step - accuracy: 0.0857 - loss: 3.9780 - top3_accuracy: 0.2168 - val_accuracy: 0.0940 - val_loss: 3.0344 - val_top3_accuracy: 0.2342 - learning_rate: 3.0000e-04
Epoch 2/15
261/261 ━━━━━━━━━━━━━━━━━━━━ 32s 121ms/step - accuracy: 0.0907 - loss: 2.9844 - top3_accuracy: 0.2249 - val_accuracy: 0.0960 - val_loss: 2.9532 - val_top3_accuracy: 0.2898 - learning_rate: 3.0000e-04
Epoch 3/15
261/261 ━━━━━━━━━━━━━━━━━━━━ 31s 118ms/step - accuracy: 0.1052 - loss: 3.3023 - top3_accuracy: 0.2595 - val_accuracy: 0.0422 - val_loss: 3.7208 - val_top3_accuracy: 0.1267 - learning_rate: 3.0000e-04
Epoch 4/15
261/261 ━━━━━━━━━━━━━━━━━━━━ 32s 121ms/step - accuracy: 0.0771 - loss: 3.4156 - top3_accuracy: 0.2404 - val_accuracy: 0.2150 - val_loss: 4.5587 - val_top3_accuracy: 0.4415 - learning_rate: 3.0000e-04
Epoch 5/15
261/261 ━━━━━━━━━━━━━━━━━━━━ 31s 120ms/step - accuracy: 0.1327 - loss: 3.1994 - top3_accuracy: 0.3064 -

In [2]:
# ===============================
# HAM10000 + MobileNetV3 (PHONE OPTIMIZED)
# ===============================

# ---- IMPORTS ----
import tensorflow as tf
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers
from google.colab import drive

print("TensorFlow version:", tf.__version__)

# ---- MOUNT DRIVE ----
drive.mount("/content/drive")

# ---- CONFIG ----
DATA_ROOT = Path("/content/drive/MyDrive/HAM10000")
IMG_DIRS = [
    DATA_ROOT / "HAM10000_images_part_1",
    DATA_ROOT / "HAM10000_images_part_2",
]
CSV_PATH = DATA_ROOT / "HAM10000_metadata.csv"

IMG_SIZE = 224
BATCH_SIZE = 32        # MobileNet allows larger batch
EPOCHS = 20            # Prevent overfitting
SEED = 42

# ---- LOAD METADATA ----
df = pd.read_csv(CSV_PATH)[["image_id", "dx"]]
print("\nClass distribution:\n", df["dx"].value_counts())

# ---- ENCODE LABELS ----
class_names = sorted(df["dx"].unique())
class_to_index = {c: i for i, c in enumerate(class_names)}
df["label"] = df["dx"].map(class_to_index)
num_classes = len(class_names)

print("\nClasses:", class_names)

# ---- RESOLVE IMAGE PATHS ----
def find_image_path(image_id):
    for d in IMG_DIRS:
        p = d / f"{image_id}.jpg"
        if p.exists():
            return str(p)
    return None

df["image_path"] = df["image_id"].apply(find_image_path)
df = df.dropna()
print("Total usable images:", len(df))

# ---- TRAIN / VAL SPLIT ----
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=SEED
)

# ---- SAFE IMAGE LOADER ----
def safe_load_image(path, label):
    def _load(path_str):
        try:
            img = tf.io.read_file(path_str)
            img = tf.image.decode_jpeg(img, channels=3)
            img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
            img = tf.cast(img, tf.float32)
            return img
        except:
            return tf.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32)

    img = tf.py_function(_load, [path], tf.float32)
    img.set_shape((IMG_SIZE, IMG_SIZE, 3))
    return img, label

# ---- MOBILE PHONE AUGMENTATION ----
augmenter = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.3),
    layers.RandomZoom(0.5),
    layers.RandomTranslation(0.25, 0.25),
    layers.RandomContrast(0.6),
    layers.RandomBrightness(0.6),
    layers.GaussianNoise(0.2),
])

def random_crop_like_phone(img):
    crop_frac = tf.random.uniform([], 0.6, 1.0)
    h = tf.shape(img)[0]
    w = tf.shape(img)[1]
    ch = tf.cast(tf.cast(h, tf.float32) * crop_frac, tf.int32)
    cw = tf.cast(tf.cast(w, tf.float32) * crop_frac, tf.int32)
    img = tf.image.random_crop(img, (ch, cw, 3))
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return img

def augment_for_phone(img, label):
    img = random_crop_like_phone(img)
    img = augmenter(img, training=True)
    return img, label

# ---- DATASET PIPELINE ----
def make_dataset(dataframe, training=True):
    ds = tf.data.Dataset.from_tensor_slices(
        (dataframe["image_path"].values, dataframe["label"].values)
    )
    ds = ds.map(safe_load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.ignore_errors()

    if training:
        ds = ds.map(augment_for_phone, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.shuffle(1024)

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)

# ---- MODEL (MobileNetV3Large) ----
base = keras.applications.MobileNetV3Large(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base.trainable = True  # FULL FINE-TUNING (SAFE FOR MOBILE)

inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
x = keras.applications.mobilenet_v3.preprocess_input(inputs)
x = base(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.summary()

# ---- COMPILE ----
model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        "accuracy",
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_accuracy")
    ]
)

# ---- CALLBACKS ----
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_top3_accuracy",
        patience=4,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )
]

# ---- TRAIN ----
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

# ---- FINAL EVALUATION ----
print("\nFinal validation metrics:")
results = model.evaluate(val_ds, verbose=0)
for n, v in zip(model.metrics_names, results):
    print(f"{n}: {v:.4f}")

# ---- SAVE MODEL ----
model.save("/content/drive/MyDrive/ham10000_mobilenetv3_phone.keras")
print("\n✅ MobileNetV3 PHONE-OPTIMIZED MODEL SAVED")


TensorFlow version: 2.19.0
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Class distribution:
 dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64

Classes: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
Total usable images: 10015
12683000/12683000 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Large (Functional)   │ (None, 7, 7, 960)      │     2,996,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 960)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 960)            │         3,840 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 960)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 7)              │         6,727 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,006,919 (11.47 MB)

 Trainable params: 2,980,599 (11.37 MB)

 Non-trainable params: 26,320 (102.81 KB)

Epoch 1/20
    251/Unknown 361s 998ms/step - accuracy: 0.2632 - loss: 2.6939 - top3_accuracy: 0.5527

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


251/251 ━━━━━━━━━━━━━━━━━━━━ 443s 1s/step - accuracy: 0.2636 - loss: 2.6924 - top3_accuracy: 0.5530 - val_accuracy: 0.4703 - val_loss: 1.8227 - val_top3_accuracy: 0.7129 - learning_rate: 1.0000e-04
Epoch 2/20
251/251 ━━━━━━━━━━━━━━━━━━━━ 43s 153ms/step - accuracy: 0.5432 - loss: 1.6851 - top3_accuracy: 0.7870 - val_accuracy: 0.6291 - val_loss: 1.2817 - val_top3_accuracy: 0.8008 - learning_rate: 1.0000e-04
Epoch 3/20
251/251 ━━━━━━━━━━━━━━━━━━━━ 43s 151ms/step - accuracy: 0.6486 - loss: 1.3620 - top3_accuracy: 0.8579 - val_accuracy: 0.6965 - val_loss: 1.0250 - val_top3_accuracy: 0.8642 - learning_rate: 1.0000e-04
Epoch 4/20
251/251 ━━━━━━━━━━━━━━━━━━━━ 43s 151ms/step - accuracy: 0.6786 - loss: 1.1435 - top3_accuracy: 0.8927 - val_accuracy: 0.6990 - val_loss: 1.0348 - val_top3_accuracy: 0.8712 - learning_rate: 1.0000e-04
Epoch 5/20
251/251 ━━━━━━━━━━━━━━━━━━━━ 43s 151ms/step - accuracy: 0.6799 - loss: 1.1279 - top3_accuracy: 0.9039 - val_accuracy: 0.7279 - val_loss: 0.8686 - val_top3_acc